# 01. 데이터셋 만들기 → 특징 추출 → 모델 비교 (Colab)

<a href="https://colab.research.google.com/github/2026-2nd/26-2-1/blob/main/notebooks/01_pipeline_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

기획서 6~8장의 절차를 그대로 실행한다. **런타임 → 런타임 유형 변경 → GPU** 를 먼저 켠다.

1. 라벨 zip → `objects.csv` (박스 1개 = 1행)
2. 세부 라벨별 표본 추출 → 수집방법·촬영일 그룹 분할(70:15:15) → 224×224 crop → `manifest.csv`
3. 특징: 수작업(색상·HOG) / 사전학습 ResNet18 512차원
4. 분류기: Dummy · 로지스틱 · SVM · 랜덤포레스트 (+ 확률 보정, τ)
5. ResNet18 미세조정

test 평가(`--final`)는 설정을 확정한 뒤 한 번만 실행한다.

데이터 준비: `00_aihub_download.ipynb`로 `labels_all`과 원천 이미지(예: `indoor_train_img`, `indoor_val_img`)를 Drive에 받아 둔다.

In [ ]:
from google.colab import drive
import os, subprocess, sys
drive.mount('/content/drive')
REPO_DIR = '/content/drive/MyDrive/ML_TeamWork/26-2-1'   # Drive에 복제한 저장소 (Drive에서 코드가 보임)
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/2026-2nd/26-2-1.git', REPO_DIR], check=True)
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
# Colab 기본 패키지(numpy·pandas·scikit-learn·opencv·scikit-image·torch)를 그대로 쓴다. 실제 버전은 실험 기록에 자동 저장된다.
!python -c "import sklearn, skimage, cv2, torch; print(sklearn.__version__, skimage.__version__, cv2.__version__, torch.__version__)"

# 원천 zip·결과는 Drive에 둔다. crop 이미지가 많아 Drive 대신 /content 에 두고 싶으면 아래 PROCESSED 단계 참고.
os.environ['ML_TEAMWORK_DATA'] = '/content/drive/MyDrive/ML_TeamWork/data'
os.environ['ML_TEAMWORK_AUTHOR'] = '이름'   # 실험 기록의 작성자
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1) 라벨 색인 (zip을 풀지 않고 읽음)
!python -m src.dataset index

In [ ]:
# 2) 표본 추출 + 그룹 분할 + crop. 받은 원천 이미지 zip에 있는 사진만 사용한다.
#    예: 실내형분류기만 → --env 실내형분류기
!python -m src.dataset build --per-code 1000 --seed 42
!cat "$ML_TEAMWORK_DATA/processed/crops_v1/split_meta.json"

In [ ]:
# 3) 특징 추출
!python -m src.extract_features --kind hand
!python -m src.extract_features --kind resnet18

In [ ]:
# 4) 특징 × 분류기 비교 (val 기준). 결과: reports/experiments/<run-id>/
!python -m src.train_classical --features hand --feature-group color
!python -m src.train_classical --features hand
!python -m src.train_classical --features resnet18

In [ ]:
# 5) ResNet18 미세조정 (GPU)
!python -m src.finetune --epochs 8 --workers 2

In [ ]:
# 6) 결과 요약 (val)
import json, glob, pandas as pd
rows = []
for p in sorted(glob.glob('reports/experiments/*/record.json')):
    r = json.load(open(p, encoding='utf-8'))
    res = r['results']
    models = res.items() if 'val' not in res else [('finetune', res)]
    for name, m in models:
        v = m.get('val_calibrated_oof') or m['val']
        rows.append({'run': r['run_id'], 'model': name, 'macro_f1': v['macro_f1'], 'misguide': v['misguide_rate'],
                     'accuracy': v['accuracy'], 'battery_recall': v.get('battery_recall'), 'ece': v.get('ece'),
                     'tau': (m.get('tau') or {}).get('tau'), 'coverage@tau': (m.get('tau') or {}).get('coverage')})
pd.DataFrame(rows).round(3)

## 결과를 GitHub에 올리기
`reports/experiments/<run-id>/`의 record.json과 그림만 커밋한다 (원본·crop 이미지·모델 파일은 올리지 않음). 작업 브랜치(`exp/...`)에서 PR로 올린다.